# 0.1 — Data pull: per-block calldata + block headers from Xatu

Produces **one row per block** for the EIP-7999 / EIP-8037 simulator. What it does:

1. Connects to the EthPandaOps ClickHouse with your credentials.
2. **Checks the schema** of the execution and beacon payload tables.
3. Pulls canonical beacon payload rows so transaction counts match the execution payload.
4. Aggregates `canonical_beacon_block_execution_transaction` **server-side** into per-block calldata bytes.
5. Pulls block headers (`gas_used`, `gas_limit`, `base_fee_per_gas`, timestamp) from `canonical_execution_block`.
6. Merges them and writes a parquet/csv with the simulator's block-level schema.

**Stubbed (filled later):** `state_gas_used` (needs state-creation bytes x CPSB — traces / Maria's repo) and `bal_bytes` (Maria's BAL estimator). The EIP-7623 calldata *floor* is a per-transaction effect and is noted as a downstream step, not reproduced here.

> Calldata lives in transactions, not block headers — so block-level calldata is *derived*. For full transaction coverage, use the canonical beacon execution-payload transaction table.


## 0. Setup

```bash
pip install -r ../requirements.txt
```

Store credentials in a local repo-root `.env` file. Do not hard-code them in the notebook:

```bash
cp ../.env.example ../.env
# edit ../.env with your real values
```


In [ ]:
import os
from pathlib import Path

import pandas as pd
import clickhouse_connect
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
load_dotenv(PROJECT_ROOT / ".env")

missing = [name for name in ["CLICKHOUSE_USER", "CLICKHOUSE_PASSWORD"] if not os.environ.get(name)]
if missing:
    raise RuntimeError(
        "Missing ClickHouse credentials: "
        + ", ".join(missing)
        + ". Create .env from .env.example at the repo root."
    )

# EthPandaOps ClickHouse HTTPS endpoint (documented). If your access uses a
# different host (e.g. a CBT/xatu-cbt endpoint), swap it here.
CH_HOST = "clickhouse-raw.xatu.ethpandaops.io"

client = clickhouse_connect.get_client(
    host=CH_HOST,
    port=443,
    secure=True,
    username=os.environ["CLICKHOUSE_USER"],
    password=os.environ["CLICKHOUSE_PASSWORD"],
)

# Smoke test
print(client.query("SELECT version()").result_rows)


## 1. Confirm the schema FIRST

Don't trust column names from memory — run these and check. In particular verify:

- `canonical_beacon_block` has `execution_payload_block_number` and `execution_payload_transactions_count`;
- `canonical_beacon_block_execution_transaction` has `slot`, `position`, and `call_data_size`;
- the **timestamp** and **base-fee** column names on `canonical_execution_block`;
- whether a **`meta_network_name`** column exists (drop the network filter if not).


In [ ]:
print("=== canonical_beacon_block ===")
display(client.query_df("DESCRIBE TABLE default.canonical_beacon_block"))

print("=== canonical_beacon_block_execution_transaction ===")
display(client.query_df("DESCRIBE TABLE default.canonical_beacon_block_execution_transaction"))

print("=== canonical_execution_block ===")
display(client.query_df("DESCRIBE TABLE default.canonical_execution_block"))


## 2. Parameters

Start with a **small block range**. A first run on one block or ~1k blocks confirms correctness before you scale up, then iterate in monthly chunks.


In [ ]:
NETWORK = "mainnet"
START_BLOCK = 21_000_000
END_BLOCK   = 21_000_000   # one-block smoke test; widen once happy

params = {"network": NETWORK, "start_block": START_BLOCK, "end_block": END_BLOCK}


## 3. Per-block calldata aggregation (server-side)

Key correctness points:

- **Use the beacon execution-payload table for raw-byte coverage.** `canonical_beacon_block_execution_transaction.call_data_size` is the baseline for total raw calldata bytes because it covers the full execution payload.
- **Use `execution_transaction` for zero/nonzero calldata gas only after validation.** It has `n_input_bytes`, `n_input_zero_bytes`, and `n_input_nonzero_bytes`; `notebooks/0.2-calldata-xatu.ipynb` checks that its count and byte total match the beacon payload before using those gas columns.
- **Aggregate on the server.** We return per-block sums, never raw transaction rows, so the result is one row per block.


In [ ]:
beacon_block_sql = '''
SELECT
    slot,
    execution_payload_block_number AS block_number,
    execution_payload_transactions_count AS n_txs_from_payload,
    execution_payload_gas_used AS gas_used_from_payload
FROM default.canonical_beacon_block FINAL
WHERE meta_network_name = {network:String}
  AND execution_payload_block_number BETWEEN {start_block:UInt64} AND {end_block:UInt64}
ORDER BY block_number
'''

df_beacon_block = client.query_df(beacon_block_sql, parameters=params)
print(df_beacon_block.shape)
df_beacon_block.head()


In [ ]:
if df_beacon_block.empty:
    raise RuntimeError("No canonical beacon blocks found for the requested execution block range")

slot_params = {
    "network": NETWORK,
    "start_slot": int(df_beacon_block["slot"].min()),
    "end_slot": int(df_beacon_block["slot"].max()),
}

calldata_sql = '''
SELECT
    slot,
    count() AS n_txs,
    sum(call_data_size) AS calldata_bytes
FROM default.canonical_beacon_block_execution_transaction FINAL
WHERE meta_network_name = {network:String}
  AND slot BETWEEN {start_slot:UInt64} AND {end_slot:UInt64}
GROUP BY slot
ORDER BY slot
'''

df_calldata_by_slot = client.query_df(calldata_sql, parameters=slot_params)
df_calldata = df_beacon_block.merge(df_calldata_by_slot, on="slot", how="left")
print(df_calldata.shape)
df_calldata.head()


## 4. Block headers

`gas_used` here is today's **combined** execution gas (no regular/state split exists on mainnet yet — EIP-8037 isn't live). For the 8037 baseline you'll later split out `state_gas_used`; until then treat `gas_used` as `regular_gas_used`.

> Confirm the timestamp column name (`block_date_time`? `timestamp`?) and base-fee name (`base_fee_per_gas`?) against the DESCRIBE output and edit below if needed.


In [ ]:
block_sql = '''
SELECT
    block_number,
    block_date_time   AS timestamp,        -- verify name
    gas_used,
    gas_limit,
    base_fee_per_gas                        -- verify name
FROM default.canonical_execution_block FINAL
WHERE meta_network_name = {network:String}
  AND block_number BETWEEN {start_block:UInt64} AND {end_block:UInt64}
ORDER BY block_number
'''

df_block = client.query_df(block_sql, parameters=params)
print(df_block.shape)
df_block.head()


## 5. Merge into the simulator's block-level schema

Stub `state_gas_used` and `bal_bytes` so the column layout matches the simulator now; they get filled when the state-bytes and BAL sources are ready.


In [ ]:
df = df_block.merge(df_calldata, on="block_number", how="left")

# Blocks with no transactions should still be present; fill aggregate NaNs with 0.
calldata_cols = ["n_txs", "calldata_bytes"]
df[calldata_cols] = df[calldata_cols].fillna(0).astype("int64")

# Validate that the transaction child rows match the beacon payload header count.
mismatch = df[df["n_txs"] != df["n_txs_from_payload"]]
if not mismatch.empty:
    display(mismatch[["block_number", "n_txs", "n_txs_from_payload"]].head())
    raise RuntimeError(f"Transaction count mismatch in {len(mismatch)} block(s)")

# baseline mapping + stubs for the resources not yet sourced
df["regular_gas_used"] = df["gas_used"]      # no regular/state split pre-8037
df["state_gas_used"]   = pd.NA               # TODO: state-creation bytes x CPSB
df["bal_bytes"]        = pd.NA               # TODO: Maria's BAL estimator

cols = ["block_number", "timestamp", "gas_used", "gas_limit", "base_fee_per_gas",
        "regular_gas_used", "state_gas_used",
        "n_txs", "calldata_bytes", "bal_bytes"]
df = df[cols].sort_values("block_number").reset_index(drop=True)
df.head()


## 6. Sanity checks


In [ ]:
print(df[["gas_used", "n_txs", "calldata_bytes"]].describe())

import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(df["block_number"], df["calldata_bytes"]); ax[0].set_ylabel("calldata bytes")
ax[1].plot(df["block_number"], df["base_fee_per_gas"]); ax[1].set_ylabel("base fee (wei)")
ax[1].set_xlabel("block_number")
plt.tight_layout(); plt.show()


## 7. Save


In [ ]:
import os
os.makedirs("data", exist_ok=True)
out = f"data/blocks_{NETWORK}_{START_BLOCK}_{END_BLOCK}"
df.to_parquet(out + ".parquet", index=False)
df.to_csv(out + ".csv", index=False)
print("wrote", out + ".parquet")


## Next steps / TODO

- **Calldata gas.** Use `notebooks/0.2-calldata-xatu.ipynb`: raw bytes come from the full-coverage beacon execution-payload table, while zero/nonzero byte counts come from `execution_transaction` after validating that it matches the beacon payload count and raw byte total.
- **`state_gas_used`.** Needs state-*creation* bytes per block (new accounts x120, new slots x64, code length), then x `CPSB`. Not in the block/tx tables — comes from traces / state-diffs, or directly from Maria's `evm-gas-repricings` repo (which already derives this). Reuse her query rather than rebuilding it.
- **`bal_bytes`.** From the RPC BAL builder in `notebooks/0.3-rpc-bal-rlp.ipynb`.
- **Scale up.** Once the small range looks right, widen the block range and pull in monthly chunks. For very large chunks, query in slot/block windows so the transaction aggregation remains bounded.
